# S4 · Проверка модели на новых сообщениях

Исторический реальный корпус UCI SMS Spam Collection, Almeida и Hidalgo, [DOI 10.24432/C5CC84](https://doi.org/10.24432/C5CC84), CC BY 4.0. Файл уже находится в `data/`, сеть не нужна.

Основной маршрут: 75 минут. Объект - одно сообщение. Повторы удаляются до разбиения. Словарь free, win, prize задан заранее. Параметры обучаются на обучающей части, порог выбирается на валидации. Блок 6 с тестом выполняется после фиксации решения. Условная цена заранее принята равной 5 × FP + FN.

Отправители и кампании неизвестны, похожие шаблоны могут оставаться. Этот опыт оценивает работу на отложенной части исторического корпуса.

### Подготовлено: импорты и чтение файла

In [1]:
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data_path = Path("data/sms-spam.tsv")
if not data_path.exists():
    data_path = Path("../data/sms-spam.tsv")
if not data_path.exists():
    data_path = Path("../../data/sms-spam.tsv")
raw_rows = []
for line in data_path.read_text(encoding="utf-8").splitlines():
    label, message = line.split("\t", 1)
    raw_rows.append((label, message))
print("Строк в исходном файле:", len(raw_rows))


Строк в исходном файле: 5574


## 1. Проверка данных и удаление повторов

`seen` хранит уже встретившиеся нормализованные тексты. `continue` переходит к следующей строке. После очистки: 5159 сообщений, из них 642 спам.

In [2]:
# Блок 1. Проверка данных и удаление повторов

messages = []
labels = []
seen = set()

for label, message in raw_rows:
    assert label in ["ham", "spam"]
    key = " ".join(message.lower().split())
    assert key != ""
    if key in seen:
        continue
    seen.add(key)
    messages.append(message)
    labels.append(int(label == "spam")) # True -> 1, False -> 0

print("Messages:", len(messages))
print("Spam:", sum(labels))
print("Not spam (ham):", len(labels) - sum(labels))

Messages: 5159
Spam: 642
Not spam (ham): 4517


### Подготовлено: тот же признак, что на S3

In [3]:
def count_words(message):
    count = 0
    for word in message.lower().split():
        word = word.strip(".,!?;:")
        if word in ["free", "win", "prize"]:
            count = count + 1
    return count


## 2. Таблица признаков и три части данных

В X одна строка содержит список из одного признака. `ids` - номера строк. `stratify` сохраняет близкие доли классов, `random_state` фиксирует разбиение. Размеры: 3095 / 1032 / 1032.

In [4]:
# Блок 2. Таблица признаков и три части данных
X = []
for message in messages:
    X.append([count_words(message)])
X = np.array(X)
y = np.array(labels)
ids = np.arange(len(y))

train_ids, other_ids = train_test_split(ids, test_size=0.4, random_state=25, stratify=y)
validation_ids, test_ids = train_test_split(other_ids, test_size=0.5, random_state=25, stratify=y[other_ids])

print("X shape:", X.shape, "y shape:", y.shape)
print("Train / validation / test:", len(train_ids), len(validation_ids), len(test_ids))

X shape: (5159, 1) y shape: (5159,)
Train / validation / test: 3095 1032 1032


## 3. Обучение библиотечной модели

`C=inf` отключает регуляризацию, чтобы сохранить функцию потерь из L1. Используется библиотечный оптимизатор L-BFGS. `max_iter` - предел его итераций. Две колонки predict_proba соответствуют classes_=[0,1]; [:,1] берёт вторую.

In [6]:
# Блок 3. Обучение библиотечной модели
model = LogisticRegression(C=float("inf"), max_iter=1000)
model.fit(X[train_ids], y[train_ids])
validation_probability = model.predict_proba(X[validation_ids])[:, 1]
print("Classes:", model.classes_)
print("w =", round(model.coef_[0, 0], 3), "b =", round(model.intercept_[0], 3))
print("First five values:", np.round(validation_probability[:5], 3))

Classes: [0 1]
w = 3.433 b = -2.386
First five values: [0.084 0.74  0.084 0.084 0.084]


## 4. Два вида ошибок и точка отсчёта

Считаем случаи по одному. TN - обычное во входящих; FP - обычное в спаме; FN - спам во входящих; TP - спам в спаме. Сумма равна числу сообщений.

In [9]:
# Блок 4. Два вида ошибок и точка отсчёта
def count_errors(actual, probability, threshold):
    fp, fn, tp, tn = 0, 0, 0, 0
    for i in range(len(actual)):
        predicted = int(probability[i] >= threshold)
        if actual[i] == 0 and predicted == 1:
            fp = fp + 1
        elif actual[i] == 1 and predicted == 0:
            fn = fn + 1
        elif actual[i] == 1 and predicted == 1:
            tp = tp + 1
        else:
            tn = tn + 1
    return tn, fp, fn, tp

always_zero = np.zeros(len(validation_ids))
baseline = count_errors(y[validation_ids], always_zero, 0.5)
print(baseline)
print(count_errors(y[validation_ids], validation_probability, 0.5))

(903, 0, 129, 0)
(889, 14, 73, 56)


## 5. Выбор порога на валидации

Выполняем только четыре заранее заданных сравнения на валидации. При одинаковой цене сохраняется первый порог. Фиксируем 0.75.

In [10]:
# Блок 5. Выбор порога на валидации
best_cost = float("inf")
best_threshold = None
validation_results = []

for threshold in [0.1, 0.25, 0.5, 0.75]:
    tn, fp, fn, tp = count_errors(y[validation_ids], validation_probability, threshold)
    cost = 5 * fp + fn
    validation_results.append([threshold, tn, fp, fn, tp, cost])
    print("T:", threshold, "FP:", fp, "FN:", fn, "Cost:", cost)
    if cost < best_cost:
        best_cost = cost
        best_threshold = threshold

    print(best_threshold)

T: 0.1 FP: 14 FN: 73 Cost: 143
0.1
T: 0.25 FP: 14 FN: 73 Cost: 143
0.1
T: 0.5 FP: 14 FN: 73 Cost: 143
0.1
T: 0.75 FP: 1 FN: 114 Cost: 119
0.75


### Подготовлено: реальные ошибки только на валидации

In [11]:
for error_name, true_label, predicted_label in [("FP", 0, 1), ("FN", 1, 0)]:
    for i in range(len(validation_ids)):
        row_id = validation_ids[i]
        prediction = int(validation_probability[i] >= best_threshold)
        if y[row_id] == true_label and prediction == predicted_label:
            print(error_name, "x =", X[row_id, 0], "p =", round(validation_probability[i], 3))
            print(messages[row_id])
            break


FP x = 2 p = 0.989
Fighting with the world is easy, u either win or lose bt fightng with some1 who is close to u is dificult if u lose - u lose if u win - u still lose.
FN x = 1 p = 0.74
Get ur 1st RINGTONE FREE NOW! Reply to this msg with TONE. Gr8 TOP 20 tones to your phone every week just £1.50 per wk 2 opt out send STOP 08452810071 16


## 6. Единственная итоговая проверка

Все решения приняты до этого блока. На этой версии данных знаменатели precision и recall положительны. В общем случае при нулевом знаменателе доля не определена и требует явной договорённости. Результат: 9 верных срабатываний из 9, найдено 9 из 128 сообщений спама. Изменения после просмотра теста требуют новой независимой проверки.

In [12]:
# Блок 6. Единственная итоговая проверка
test_probability = model.predict_proba(X[test_ids])[:, 1]
tn, fp, fn, tp = count_errors(y[test_ids], test_probability, best_threshold)
accuracy = (tp + tn) / len(test_ids)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
test_cost = 5 * fp + fn
test_baseline_cost = int(sum(y[test_ids]))
print("TN FP FN TP =", tn, fp, fn, tp)
print("Accuracy", round(accuracy, 3))
print("precision", round(precision, 3))
print("recall", round(recall, 3))
print(test_cost, test_baseline_cost)

TN FP FN TP = 904 0 119 9
Accuracy 0.885
precision 1.0
recall 0.07
119 128
